# 01 — Data Collection từ DBpedia SPARQL
> **Chương 6.1 — WikiCities Dataset**  
> Thu thập dữ liệu các thành phố từ DBpedia Knowledge Graph thông qua SPARQL endpoint,
> bao gồm các thuộc tính địa lý, hành chính và ngữ nghĩa để dự đoán dân số.


## 0. Cài đặt thư viện

In [15]:
import os
print(os.getcwd())

/content/Seminar-Pattern-Recognition


In [16]:
# ============================================================
# Cell 0: Clone repo vào Colab (chạy MỖI lần khởi động runtime)
# ============================================================
import os, subprocess

REPO_URL = 'https://github.com/hoagannhh/Seminar-Pattern-Recognition.git'
REPO_DIR = '/content/Seminar-Pattern-Recognition'

if not os.path.exists(REPO_DIR):
    print('Cloning repo...')
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
    print('Clone xong!')
else:
    print('Repo đã tồn tại, pulling latest changes...')
    subprocess.run(['git', 'pull'], cwd=REPO_DIR, check=True)

print(f'Repo dir: {REPO_DIR}')
# Chạy cell này ĐẦU TIÊN mỗi session
%run /content/Seminar-Pattern-Recognition/setup_colab.py


Repo đã tồn tại, pulling latest changes...
Repo dir: /content/Seminar-Pattern-Recognition
🔄 Repo đã có, pulling latest...
✅ Pull xong
📁 Working dir: /content/Seminar-Pattern-Recognition
🐍 src/ đã thêm vào sys.path
📦 Cài thư viện...
✅ Cài xong
📂 Thư mục data/ models/ reports/ sẵn sàng
🖥️  GPU: Tesla T4, 15360 MiB

🚀 Setup hoàn tất! Bắt đầu chạy notebook.


In [17]:
import time
import math
import requests
import pandas as pd
from pathlib import Path
from tqdm.notebook import tqdm
from SPARQLWrapper import SPARQLWrapper, JSON

# Tạo thư mục lưu data nếu chưa có
RAW_DIR = Path('../data/raw')
RAW_DIR.mkdir(parents=True, exist_ok=True)

print(' Thư viện đã sẵn sàng')
print(f' Lưu dữ liệu tại: {RAW_DIR.resolve()}')

 Thư viện đã sẵn sàng
 Lưu dữ liệu tại: /content/data/raw


## 1. Cấu hình SPARQL Endpoint

In [18]:
SPARQL_ENDPOINT = 'https://dbpedia.org/sparql'
PAGE_SIZE       = 1000   # số thành phố mỗi lần query (tránh timeout)
MAX_CITIES      = 10000  # giới hạn tổng (None = lấy tất cả)
SLEEP_SEC       = 2.0    # nghỉ giữa các request (tránh bị block)

sparql = SPARQLWrapper(SPARQL_ENDPOINT)
sparql.setReturnFormat(JSON)
sparql.setTimeout(60)

print(f' Endpoint : {SPARQL_ENDPOINT}')
print(f' Page size: {PAGE_SIZE} | Max cities: {MAX_CITIES}')

 Endpoint : https://dbpedia.org/sparql
 Page size: 1000 | Max cities: 10000


## 2. Truy vấn SPARQL chính

Lấy các thuộc tính quan trọng của từng thành phố:

| Feature | Thuộc tính DBpedia | Nhóm |
|---|---|---|
| `name` | `rdfs:label` | ID |
| `population` | `dbo:populationTotal` | **Target** |
| `area` | `dbo:areaTotal` | Địa lý |
| `elevation` | `dbo:elevation` | Địa lý |
| `lat`, `lon` | `geo:lat`, `geo:long` | Địa lý |
| `country` | `dbo:country` | Hành chính |
| `region` | `dbo:region` | Hành chính |
| `populationDensity` | `dbo:populationDensity` | Tính toán |
| `foundingYear` | `dbo:foundingYear` | Lịch sử |
| `abstract` | `dbo:abstract` (EN) | NLP |
| `numTriples` | `COUNT(triples)` | Graph structure |


In [19]:
QUERY_TEMPLATE = """
PREFIX dbo:  <http://dbpedia.org/ontology/>
PREFIX dbp:  <http://dbpedia.org/property/>
PREFIX geo:  <http://www.w3.org/2003/01/geo/wgs84_pos#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

SELECT DISTINCT
  ?city
  (STR(?nameRaw)        AS ?name)
  (?population          AS ?population)
  (?area                AS ?area)
  (?elevation           AS ?elevation)
  (?lat                 AS ?lat)
  (?lon                 AS ?lon)
  (STR(?countryRaw)     AS ?country)
  (STR(?regionRaw)      AS ?region)
  (?popDensity          AS ?populationDensity)
  (?foundingYear        AS ?foundingYear)
  (STR(?abstractRaw)    AS ?abstract)
WHERE {{
  ?city a dbo:City .
  ?city rdfs:label           ?nameRaw .
  ?city dbo:populationTotal  ?population .

  OPTIONAL {{ ?city dbo:areaTotal         ?area         }}
  OPTIONAL {{ ?city dbo:elevation         ?elevation    }}
  OPTIONAL {{ ?city geo:lat               ?lat          }}
  OPTIONAL {{ ?city geo:long              ?lon          }}
  OPTIONAL {{ ?city dbo:country           ?countryRaw   }}
  OPTIONAL {{ ?city dbo:region            ?regionRaw    }}
  OPTIONAL {{ ?city dbo:populationDensity ?popDensity   }}
  OPTIONAL {{ ?city dbo:foundingYear      ?foundingYear }}
  OPTIONAL {{ ?city dbo:abstract          ?abstractRaw .
              FILTER(lang(?abstractRaw) = 'en') }}

  FILTER(lang(?nameRaw) = 'en')
  FILTER(?population > 1000)
}}
ORDER BY DESC(?population)
LIMIT  {limit}
OFFSET {offset}
"""

print(' Query template đã sẵn sàng')

 Query template đã sẵn sàng


## 3. Query đếm tổng số thành phố

In [20]:
COUNT_QUERY = """
PREFIX dbo:  <http://dbpedia.org/ontology/>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

SELECT (COUNT(DISTINCT ?city) AS ?total)
WHERE {
  ?city a dbo:City .
  ?city rdfs:label          ?name .
  ?city dbo:populationTotal ?population .
  FILTER(lang(?name) = 'en')
  FILTER(?population > 1000)
}
"""

sparql.setQuery(COUNT_QUERY)
result = sparql.query().convert()
total_cities = int(result['results']['bindings'][0]['total']['value'])

if MAX_CITIES:
    total_cities = min(total_cities, MAX_CITIES)

total_pages = math.ceil(total_cities / PAGE_SIZE)
print(f'  Tổng số thành phố sẽ thu thập: {total_cities:,}')
print(f' Số lượt query (pages)         : {total_pages}')

  Tổng số thành phố sẽ thu thập: 10,000
 Số lượt query (pages)         : 10


## 4. Thu thập dữ liệu theo từng trang (pagination)

In [21]:
def fetch_page(offset: int, limit: int) -> list[dict]:
    """Gửi 1 SPARQL request và trả về list of dicts."""
    query = QUERY_TEMPLATE.format(limit=limit, offset=offset)
    sparql.setQuery(query)
    result = sparql.query().convert()
    rows = []
    for b in result['results']['bindings']:
        row = {
            'uri'               : b.get('city',              {}).get('value'),
            'name'              : b.get('name',              {}).get('value'),
            'population'        : b.get('population',        {}).get('value'),
            'area'              : b.get('area',              {}).get('value'),
            'elevation'         : b.get('elevation',         {}).get('value'),
            'lat'               : b.get('lat',               {}).get('value'),
            'lon'               : b.get('lon',               {}).get('value'),
            'country'           : b.get('country',           {}).get('value'),
            'region'            : b.get('region',            {}).get('value'),
            'populationDensity' : b.get('populationDensity', {}).get('value'),
            'foundingYear'      : b.get('foundingYear',      {}).get('value'),
            'abstract'          : b.get('abstract',          {}).get('value'),
        }
        rows.append(row)
    return rows


all_rows = []
failed_pages = []

for page in tqdm(range(total_pages), desc='Fetching DBpedia'):
    offset = page * PAGE_SIZE
    limit  = min(PAGE_SIZE, total_cities - offset)
    try:
        rows = fetch_page(offset, limit)
        all_rows.extend(rows)
        time.sleep(SLEEP_SEC)
    except Exception as e:
        print(f'  Page {page} (offset={offset}) lỗi: {e}')
        failed_pages.append(page)
        time.sleep(SLEEP_SEC * 3)  # nghỉ lâu hơn khi lỗi

print(f'\n Thu thập xong: {len(all_rows):,} bản ghi')
if failed_pages:
    print(f'  Pages thất bại: {failed_pages} — chạy lại ô bên dưới')

Fetching DBpedia:   0%|          | 0/10 [00:00<?, ?it/s]


 Thu thập xong: 10,000 bản ghi


## 5. Retry các trang bị lỗi (nếu có)

In [22]:
if failed_pages:
    print(f' Retry {len(failed_pages)} pages...')
    still_failed = []
    for page in tqdm(failed_pages, desc='Retrying'):
        offset = page * PAGE_SIZE
        limit  = min(PAGE_SIZE, total_cities - offset)
        try:
            rows = fetch_page(offset, limit)
            all_rows.extend(rows)
            time.sleep(SLEEP_SEC * 2)
        except Exception as e:
            print(f' Page {page} vẫn lỗi: {e}')
            still_failed.append(page)
    failed_pages = still_failed
    print(f' Sau retry: {len(all_rows):,} bản ghi | Còn lỗi: {failed_pages}')
else:
    print(' Không có page nào cần retry')

 Không có page nào cần retry


## 6. Tạo DataFrame và làm sạch sơ bộ

In [23]:
df = pd.DataFrame(all_rows)

# ── Ép kiểu số ──────────────────────────────────────────────────────────────
NUM_COLS = ['population', 'area', 'elevation', 'lat', 'lon',
            'populationDensity', 'foundingYear']

for col in NUM_COLS:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# ── Trích xuất tên quốc gia từ URI (vd: http://dbpedia.org/resource/Vietnam) ─
df['country_name'] = df['country'].str.extract(r'/resource/(.+)$')
df['region_name']  = df['region'].str.extract(r'/resource/(.+)$')

# ── Loại trùng lặp theo URI ──────────────────────────────────────────────────
before = len(df)
df = df.drop_duplicates(subset='uri').reset_index(drop=True)
print(f' Loại trùng lặp: {before:,} → {len(df):,} thành phố')

# ── Loại dòng không có target ────────────────────────────────────────────────
df = df.dropna(subset=['population']).reset_index(drop=True)
print(f' Sau khi lọc population null: {len(df):,} thành phố')

df.head(3)

 Loại trùng lặp: 10,000 → 5,826 thành phố
 Sau khi lọc population null: 5,826 thành phố


,uri,name,population,area,elevation,lat,lon,country,region,populationDensity,foundingYear,abstract,country_name,region_name
0,http://dbpedia.org/resource/Dusmareb,Dusmareb,680000407,NaN,NaN,5.535000,46.386112,http://dbpedia.org/resource/Somalia,None,NaN,NaN,None,Somalia,NaN
1,http://dbpedia.org/resource/Rabaul,Rabaul,388517044,NaN,NaN,-4.200000,152.183334,http://dbpedia.org/resource/Papua_New_Guinea,None,NaN,NaN,None,Papua_New_Guinea,NaN
2,http://dbpedia.org/resource/Beijing,Beijing,21893095,1.641054e+10,43.5,39.906666,116.397499,None,None,NaN,NaN,None,NaN,NaN


## 7. Thêm feature: số lượng triples (graph structure)

> **Phát hiện chính từ EDA (§ 6.2):** Số lượng triples của một thành phố trong KG là **indicator tốt nhất** của dân số — thành phố lớn được mô tả nhiều hơn trong Wikipedia/DBpedia.

In [24]:
TRIPLE_COUNT_TEMPLATE = """
SELECT (COUNT(*) AS ?numTriples)
WHERE {{
  <{uri}> ?p ?o .
}}
"""

def get_triple_count(uri: str) -> int | None:
    """Đếm số triples của một entity."""
    try:
        sparql.setQuery(TRIPLE_COUNT_TEMPLATE.format(uri=uri))
        result = sparql.query().convert()
        return int(result['results']['bindings'][0]['numTriples']['value'])
    except:
        return None


# Chỉ sample 2000 thành phố để không query quá lâu
# (đủ để demo tương quan triple count vs population)
SAMPLE_FOR_TRIPLES = 2000
sample_idx = df.sample(min(SAMPLE_FOR_TRIPLES, len(df)), random_state=42).index

df['numTriples'] = None

for idx in tqdm(sample_idx, desc='Counting triples'):
    uri = df.at[idx, 'uri']
    df.at[idx, 'numTriples'] = get_triple_count(uri)
    time.sleep(0.3)

df['numTriples'] = pd.to_numeric(df['numTriples'], errors='coerce')

n_filled = df['numTriples'].notna().sum()
print(f' Đã lấy numTriples cho {n_filled:,} thành phố')

Counting triples:   0%|          | 0/2000 [00:00<?, ?it/s]

 Đã lấy numTriples cho 2,000 thành phố


## 8. Thêm feature: độ phong phú thông tin (information richness)

In [25]:
# Đếm số thuộc tính không-null của mỗi thành phố → proxy cho "mức độ được mô tả"
ATTRIBUTE_COLS = ['area', 'elevation', 'lat', 'lon', 'country',
                  'region', 'populationDensity', 'foundingYear', 'abstract']

df['numAttributes'] = df[ATTRIBUTE_COLS].notna().sum(axis=1)
df['hasAbstract']   = df['abstract'].notna().astype(int)
df['abstractLen']   = df['abstract'].fillna('').str.len()

print(' Đã thêm: numAttributes, hasAbstract, abstractLen')
df[['name', 'population', 'numAttributes', 'numTriples', 'abstractLen']].head(10)

 Đã thêm: numAttributes, hasAbstract, abstractLen


,name,population,numAttributes,numTriples,abstractLen
0,Dusmareb,680000407,3,NaN,0
1,Rabaul,388517044,3,NaN,0
2,Beijing,21893095,4,NaN,0
3,Chengdu,20937757,5,NaN,0
4,Karachi,18868021,4,NaN,0
5,Guangzhou,18676605,5,NaN,0
6,Shenzhen,17560000,4,NaN,0
7,Kinshasa,17071000,6,NaN,0
8,Delhi,16787941,5,1240.0,0
9,Tokyo,14254039,5,NaN,0


## 9. Lưu dữ liệu thô

In [26]:
out_path = RAW_DIR / 'wikicities_raw.csv'
df.to_csv(out_path, index=False)

print(f'Đã lưu: {out_path}')
print(f'Shape : {df.shape[0]:,} rows × {df.shape[1]} cols')
print()
print('Thống kê cơ bản:')
print(df[['population', 'area', 'elevation', 'numAttributes', 'numTriples']].describe().round(1))

Đã lưu: ../data/raw/wikicities_raw.csv
Shape : 5,826 rows × 18 cols

Thống kê cơ bản:
        population          area  elevation  numAttributes  numTriples
count       5826.0  4.442000e+03     3460.0         5826.0      2000.0
mean      758083.8  2.322793e+09      405.3            4.4       408.8
std     10352644.6  1.440503e+10      576.5            0.9       283.7
min        56055.0  1.331254e+06      -28.0            0.0        66.0
25%        89659.2  5.350000e+07       35.0            4.0       210.0
50%       155925.0  1.597100e+08      182.9            4.0       333.0
75%       396643.2  7.175450e+08      483.0            5.0       510.0
max    680000407.0  4.505370e+11     4330.0            6.0      2278.0


## 10. Tổng kết & Missing values

In [27]:
print('=== MISSING VALUES ===')
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(1)
summary = pd.DataFrame({'missing': missing, 'pct': missing_pct})
print(summary[summary['missing'] > 0].sort_values('pct', ascending=False).to_string())

print('\n=== TARGET DISTRIBUTION ===')
print(df['population'].describe().apply(lambda x: f'{x:,.0f}'))

print('\n=== CÁC NƯỚC NHIỀU THÀNH PHỐ NHẤT ===')
print(df['country_name'].value_counts().head(15).to_string())

=== MISSING VALUES ===
                   missing    pct
abstract              5826  100.0
foundingYear          5826  100.0
region                5768   99.0
region_name           5768   99.0
populationDensity     4606   79.1
numTriples            3826   65.7
elevation             2366   40.6
area                  1384   23.8
country                948   16.3
country_name           948   16.3
lon                    106    1.8
lat                    106    1.8

=== TARGET DISTRIBUTION ===
count          5,826
mean         758,084
std       10,352,645
min           56,055
25%           89,659
50%          155,925
75%          396,643
max      680,000,407
Name: population, dtype: object

=== CÁC NƯỚC NHIỀU THÀNH PHỐ NHẤT ===
country_name
India            778
China            442
Japan            363
United_States    265
Pakistan         159
Mexico           157
Egypt            131
Iran             120
Argentina        102
Indonesia         98
Vietnam           76
Chile             69
It

##  Kết quả

File được lưu tại `data/raw/wikicities_raw.csv` với các cột:

| Cột | Mô tả |
|---|---|
| `uri` | DBpedia URI của thành phố |
| `name` | Tên thành phố (tiếng Anh) |
| `population` | **TARGET** — dân số |
| `area` | Diện tích (m²) |
| `elevation` | Độ cao (m) |
| `lat`, `lon` | Tọa độ địa lý |
| `country_name` | Tên quốc gia (từ URI) |
| `region_name` | Tên vùng/tỉnh |
| `populationDensity` | Mật độ dân số |
| `foundingYear` | Năm thành lập |
| `abstract` | Mô tả từ Wikipedia |
| `numTriples` | Số triples trong KG (graph feature) |
| `numAttributes` | Số thuộc tính không-null |
| `hasAbstract` | 0/1 |
| `abstractLen` | Độ dài mô tả |

**Bước tiếp theo:** `02_eda.ipynb`
